In [ ]:
C:/Users/IVL-SERVER2/Desktop/jihun/mvsplat/datasets/colmap_based

## 🟦 Input Image Enhancement

##### Zero-DCE

## 🟦 Baseline Gaussian Reconstruction

### Vanilla 3DGS

**Train**

In [ ]:
python train.py -s <path to COLMAP or NeRF Synthetic dataset> --eval # Train with train/test split
python render.py -m <path to trained model> # Generate renderings
python metrics.py -m <path to trained model> # Compute error metrics on renderings

### MVSplat

##### Setup

In [ ]:
python -m pip install --retries 20 --timeout 180 --no-build-isolation --extra-index-url https://download.pytorch.org/whl/cu118 -r requirements.txt numpy==1.26.4 torch==2.1.2+cu118 torchvision==0.16.2+cu118 torchaudio==2.1.2+cu118

In [ ]:
python -c "import numpy, torch, torchvision, torchaudio; print('NumPy:', numpy.__version__); print('Torch:', torch.__version__, torch.version.cuda); print('Torchvision:', torchvision.__version__); print('Torchaudio:', torchaudio.__version__); print('CUDA available:', torch.cuda.is_available())"

### 🖼️ Loading Datasets

#### 1️⃣ Example with DTU test datasets

- Evaluating with DTU examples with re10k checkpoints

In [ ]:
python -m src.main +experiment=dtu \
checkpointing.load=checkpoints/re10k.ckpt \
mode=test \
dataset/view_sampler=evaluation \
dataset.view_sampler.index_path=assets/evaluation_index_dtu_nctx2.json \
test.save_video=true \
test.save_image=false \
test.compute_scores=false

이후 렌더링 파일은 `outputs/test/dtu/views2/video` 경로에 생성된다

##### `convert_dtu.py`

In [ ]:
def load_images(example_path: Path) -> dict[int, UInt8[Tensor, "..."]]:
    """Load JPG images as raw bytes (do not decode)."""
    images_dict = {}
    for cur_id in range(1, 50):
        cur_image_name = f"rect_{cur_id:03d}_3_r5000.png" # <- 3번 조명만 읽고 있다
        img_bin = load_raw(example_path / cur_image_name)
        images_dict[cur_id - 1] = img_bin

##### `dtu.yaml`

**평가할 때(evaluation)**

```yaml
dataset:
    ...
    test_times_per_scene: 4 # <- scene마다 4개의 시점을 eval
```

##### `mvsplat\assets\evaluation_index_dtu_nctx2.json`

장면마다 2개의 context view `[33, 31]`을 고정, target view는 `[32, 24, 23, 44]` 중 하나를 지정
- 16개의 scene x 4 target view = 64 evaluation samples
- `rect_034_3_r5000.png`, `rect_032_3_r5000.png`가 target이 됨

#### 2️⃣ Converting COLMAP-based datasets

**Github Issue Page** : https://github.com/donydchen/mvsplat/issues/1

#### Inference on a COLMAP scene

`src.scripts.infer_colmap` renders one target camera using two fixed context images.
It reads a COLMAP text model, uses the registered images sorted by filename for
zero-based view indices, and does not load the target image. If your COLMAP model is
binary, convert it to text first:

In [ ]:
colmap model_converter \
  --input_path PATH_TO_COLMAP_MODEL \
  --output_path PATH_TO_COLMAP_TEXT_MODEL \
  --output_type TXT

For undistorted `PINHOLE` or `SIMPLE_PINHOLE` cameras, run:

In [ ]:
python -m src.scripts.infer_colmap \
  --colmap-model ./datasets/colmap_based/office_undistorted/sparse \
  --images ./datasets/colmap_based/office_undistorted/images \
  --checkpoint checkpoints/re10k.ckpt \
  --context-indices 2 8 \
  --target-index 5 \
  --near 30 \
  --far 100 \
  --output outputs/colmap/office_render.png \
  --ply-output outputs/colmap/office_gaussians.ply

The context and target indices above are fixed for every invocation; indices refer
to the filename-sorted registered COLMAP images. Set `--near` and `--far` to bounds
appropriate for the COLMAP reconstruction scale. The target camera pose is used for
rendering, while its image is not required.

**🔹Gauissian Primitives (`.ply`)**

The script also exports the inferred
Gaussians as a PLY file. The PLY exporter recenters/rescales the point cloud for
viewing and writes the view-independent DC color component.

#### Possible Issues

**🔹 `SIMPLE_RADIAL` → `PINHOLE`**

In [ ]:
colmap image_undistorter \
    --image_path /path/to/project/images \
    --input_path /path/to/project/sparse/0 \
    --output_path /path/to/project/undistorted \
    --output_type COLMAP

## 🟦 Gaussian Enhance Network

- Zero-DCE의 커브 예측 네트워크의 구조를 가져오되, 가우시안 파일 `PLY`는 순서와 관계가 없고 가우시안 자체의 피처를 추출해야하므로 `Conv2d` 대신 `Conv1d`를 사용한다.
- 네트워크는 `[B, 3, N]`의 `shape`를 가진 텐서를 입력으로 받으며, 이때 `3`은 SH DC coefficient에 대응된다. 하지만 Zero-DCE의 네트워크는 SH 계수가 아니라 RGB 값에 적용되므로, RGB 값으로 변환하여 네트워크에 입력한다.

```python
C0 = 0.28209479177387814
rgb = f_dc * C0 + 0.5
...
# enhanced_rgb = output of the enhance network
f_dc_enhanced = (enhanced_rgb - 0.5) / C0
```

**초기 네트워크** : SH DC 계수만 수정하고 `f_rest`, 고차 SH 계수는 변경되지 않으므로 실험 이후 `f_rest`까지 수정하는 구조로 변경을 고려해볼 수 있다.

`opacity`의 고정 여부도 결정해야 함